# VISIONSTREAM AI — SPRINT 2 / TASK S2.4
## GATE 1: Google Colab Free GPU Feasibility Trial & 1-Epoch Execution

**Project:** VisionStream AI  
**Objective:** Verify GPU hardware, environment compatibility, dataset integrity, and execute exactly a ONE-EPOCH feasibility trial.  
**Strict Boundary:** Zero dollar expenditure ($0). Free GPU only. Sealed Holdout Test.

### Step 1: Hardware & Environment Inspection

In [ ]:
!nvidia-smi
import torch, torchvision, sys
print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"CUDA Device Capability: {torch.cuda.get_device_capability(0)}")
else:
    raise SystemExit("FATAL: No GPU available! Stop and report BLOCKED.")

### Step 2: Install Ultralytics and Record Versions

In [ ]:
!pip install ultralytics
import ultralytics
print(f"Ultralytics Version: {ultralytics.__version__}")

### Step 3: Mount Google Drive & Verify Dataset Archive

In [ ]:
from google.colab import drive
import hashlib, os, zipfile, shutil
from pathlib import Path

drive.mount('/content/drive')

# Set path to where you uploaded coco-person-sprint2.zip in your Google Drive
DRIVE_ZIP = Path('/content/drive/MyDrive/coco-person-sprint2.zip')

if not DRIVE_ZIP.exists():
    # Try search across MyDrive
    matches = list(Path('/content/drive/MyDrive').rglob('coco-person-sprint2.zip'))
    if matches:
        DRIVE_ZIP = matches[0]
    else:
        raise FileNotFoundError("Archive not found in Drive! Please upload coco-person-sprint2.zip to your Google Drive MyDrive")

print(f"Found archive at: {DRIVE_ZIP}")
archive_bytes = DRIVE_ZIP.read_bytes()
archive_sha = hashlib.sha256(archive_bytes).hexdigest()
print(f"Drive Archive Size: {len(archive_bytes)} bytes")
print(f"Drive Archive SHA256: {archive_sha}")

EXPECTED_ZIP_SHA = "ad28fb9e221d9a781cdaf88334beb93897db9898c20df0125a035aa59fa0de9e"
if archive_sha != EXPECTED_ZIP_SHA:
    raise ValueError(f"FATAL: Archive SHA256 mismatch! Expected {EXPECTED_ZIP_SHA}, got {archive_sha}")
print("Archive SHA256 MATCH VERIFIED! (100% Identical)")

### Step 4: Extract Dataset & Verify Per-Image SHA-256 Checksums

In [ ]:
EXTRACT_DIR = Path('/content/dataset/coco-person')
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print("Extracting dataset to ephemeral storage...")
with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
    zf.extractall(EXTRACT_DIR)

train_imgs = list((EXTRACT_DIR / 'images/train').glob('*.jpg'))
val_imgs = list((EXTRACT_DIR / 'images/val').glob('*.jpg'))
test_imgs = list((EXTRACT_DIR / 'images/test').glob('*.jpg'))
print(f"Extracted: Train={len(train_imgs)}, Val={len(val_imgs)}, Test={len(test_imgs)}")
assert len(train_imgs) == 2000, f"Expected 2000 train, got {len(train_imgs)}"
assert len(val_imgs) == 239, f"Expected 239 val, got {len(val_imgs)}"
assert len(test_imgs) == 0, "Holdout test leaked into training directory!"

# Checksum verification for train and val images against manifest
checksum_file = EXTRACT_DIR / 'metadata/checksums.sha256'
expected_hashes = {}
for line in checksum_file.read_text().splitlines():
    if line.strip():
        sha, rel = line.strip().split('  ')
        expected_hashes[rel] = sha

print(f"Loaded {len(expected_hashes)} canonical checksum entries from manifest.")
mismatches = 0
checked = 0
for rel, exp_sha in expected_hashes.items():
    if 'test' in rel:
        continue # test is strictly sealed
    img_path = EXTRACT_DIR / rel
    if not img_path.exists():
        mismatches += 1
        continue
    act_sha = hashlib.sha256(img_path.read_bytes()).hexdigest()
    if act_sha != exp_sha:
        mismatches += 1
    checked += 1

print(f"Checked {checked} training & validation images against canonical SHA-256: {mismatches} mismatches.")
if mismatches > 0:
    raise ValueError(f"FATAL: Checksum verification failed with {mismatches} mismatches! ABORTING.")
print("PER-IMAGE SHA-256 INTEGRITY VERIFIED (100% MATCH)!")

### Step 5: Configure dataset.yaml for Colab Paths

In [ ]:
yaml_content = f"""# Ultralytics YOLOv8 Dataset Configuration - Colab Feasibility
path: {EXTRACT_DIR}
train: images/train
val: images/val

names:
  0: person
"""
colab_yaml = EXTRACT_DIR / 'colab_dataset.yaml'
colab_yaml.write_text(yaml_content)
print("colab_dataset.yaml created:")
print(colab_yaml.read_text())

### Step 6: Verify Baseline Checkpoint & Audit Weight Transfer

In [ ]:
from ultralytics import YOLO

# Check for custom baseline checkpoint transfer or download official release
DRIVE_MODEL = Path('/content/drive/MyDrive/yolov8n.pt')
if DRIVE_MODEL.exists():
    shutil.copy(DRIVE_MODEL, 'yolov8n.pt')
    print("Copied verified baseline yolov8n.pt from Drive.")

model = YOLO('yolov8n.pt')
model_path = Path('yolov8n.pt')
if model_path.exists():
    m_sha = hashlib.sha256(model_path.read_bytes()).hexdigest()
    print(f"Baseline yolov8n.pt SHA256: {m_sha}")
    EXPECTED_MODEL_SHA = "f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d2c8412c47ccfc83b36"
    if m_sha == EXPECTED_MODEL_SHA:
        print("Baseline model SHA-256 matches exact project baseline!")
    else:
        print(f"Notice: Baseline model SHA ({m_sha}) differs from local commit ({EXPECTED_MODEL_SHA})")

### Step 7: Execute ONE-EPOCH Feasibility Trial ONLY

In [ ]:
import time

print("=== STARTING GATE 1: ONE-EPOCH FEASIBILITY TRIAL ===")
start_time = time.time()

# STRICT EXECUTION LIMIT: epochs=1 ONLY
results = model.train(
    data=str(colab_yaml),
    epochs=1,               # EXACTLY ONE EPOCH
    imgsz=640,
    batch=16,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=0,        # No warmup needed for 1-epoch trial
    seed=42,
    amp=True,
    workers=4,
    project='/content/runs/feasibility',
    name='gate1_trial',
    exist_ok=True,
    plots=True
)

elapsed = time.time() - start_time
print(f"=== 1-EPOCH TRIAL COMPLETED IN {elapsed:.2f} SECONDS ===")

### Step 8: Inspect VRAM Usage, Metrics & Save Evidence

In [ ]:
!nvidia-smi
import pandas as pd

run_dir = Path('/content/runs/feasibility/gate1_trial')
csv_path = run_dir / 'results.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    print("Results CSV contents:")
    print(df.to_string())

# Copy trial results to Google Drive for offline review
DEST_EVIDENCE = Path('/content/drive/MyDrive/visionstream_gate1_evidence')
DEST_EVIDENCE.mkdir(parents=True, exist_ok=True)
for f in run_dir.glob('*'):
    if f.is_file():
        shutil.copy(f, DEST_EVIDENCE / f.name)
print(f"Evidence artifacts safely copied to {DEST_EVIDENCE} for local project reporting!")